In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from pycox.models.cox import CoxPH
from pycox.evaluation import EvalSurv
import matplotlib.pyplot as plt
import shap
import scipy.integrate
scipy.integrate.simps = scipy.integrate.simpson



from itertools import product
import torch

sys.path.append(os.path.abspath("../../"))
from src.dataset.generate_dataset import TorchPreprocessing
from src.utils.kapplan_meir import k_m_cox
from src.dataset.DataSet import SurvivalDataSet
from src.utils.Preprocessing import Preprocessor
from src.utils.ConvertTextToCsv import TextToCsv
from src.dataset.split_data import split_data_Train_Val_Test
from src.utils.set_seed import set_seed
from src.utils.cox_models import *
from sklearn.utils import resample


import numpy as np
import torch
import torchtuples as tt

%load_ext autoreload
%autoreload 2
%matplotlib inline

/Users/administrador/Library/Python/3.11/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
set_seed(42)

In [3]:
pp = Preprocessor()

In [4]:
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_data = pp.clean_columns_dataset(df_clinical_data)
list_df = pp.total_type_len_type_cancer(df_clinical_data)
df_clinical_data["Tumor-Cancer"] = list_df
df_clinical_data["Tumor-Cancer"].unique()

df_mRNA_raw_data = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")


Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 299 - Total(%) 0.37
Shape of the CSV: (20440, 819)


In [42]:
df_merged = TorchPreprocessing(df_mRNA_raw_data,df_clinical_data, 23360).get_comparation_df()
 
comparation_df = df_merged.loc[
    df_merged["Tumor-Cancer"].isin(["Luminal A", "Luminal B", "TNBC", "HER2-enriched"]),
]

comparation_df["Tumor-Cancer"].unique()

print(f"Samples: {comparation_df.shape[0]}, Genes: {comparation_df.shape[1]}")

Genes before Treshold: 20434
count    20434.000000
mean        73.977831
std        155.539160
min          0.000000
25%          0.000000
50%          0.000000
75%         21.000000
max        519.000000
dtype: float64
Threshold (>80% zeros): 415 samples
After the treshold: 18533
Samples: 519, Genes: 18534


In [43]:
zero_reduced_df =  comparation_df.drop(["Sample ID"], axis=1)
results_df, desing, expr = pp.initialize_limma(zero_reduced_df, column="Tumor-Cancer", column_event="Overall Survival (Months)", column_status="Overall Survival Status")

In [44]:
N_GENES = 1000
top_genes_limma = results_df.sort_values("pvalue").index[:N_GENES].tolist()

In [45]:
torch_preprocessing = TorchPreprocessing(df_mRNA_raw_data, df_clinical_data, 23360)
torch_preprocessing.genes_expression = top_genes_limma
genes_expression = top_genes_limma
X_scaled, durations, events, scaler, sample_ids = torch_preprocessing.get_data_set(60, return_ids=True)


Genes before Treshold: 1003
count    1003.000000
mean        0.929212
std         3.489359
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        42.000000
dtype: float64
Threshold (>80% zeros): 415 samples
After the treshold: 1003


In [46]:
surival_data_set  = SurvivalDataSet(X_scaled, durations, events)

In [47]:
import json
from src.dataset.split_data import split_dataset_by_ids

# Load the canonical split created by limma.ipynb so the NN uses the SAME patients.
split = json.load(open("splits.json"))
set_seed()
(train_X, train_durations, train_events), \
(val_X, val_durations, val_events), \
(test_X, test_durations, test_events) = split_dataset_by_ids(
    surival_data_set, sample_ids,
    split["train_ids"], split["val_ids"], split["test_ids"]
)
print(f"Train: {train_X.shape[0]}, Val: {val_X.shape[0]}, Test: {test_X.shape[0]}")
print(f"Train: {type(train_X)}, Val: {type(val_X)}, Test: {type(test_X)}")

Train: 331, Val: 84, Test: 104
Train: <class 'torch.Tensor'>, Val: <class 'torch.Tensor'>, Test: <class 'torch.Tensor'>


In [48]:
in_features = train_X.shape[1]
train_X_np = train_X.cpu().numpy()
train_durations_np = train_durations.cpu().numpy()
train_events_np = train_events.cpu().numpy()



val_X_np = val_X.cpu().numpy()
val_durations_np = val_durations.cpu().numpy()
val_events_np = val_events.cpu().numpy()


test_X_np = test_X.cpu().numpy()
test_durations_np = test_durations.cpu().numpy()
test_events_np = test_events.cpu().numpy()

train_X_np.shape, train_durations_np.shape, train_events_np.shape

((331, 1000), (331,), (331,))

## DeepSurv Bootstrap

In [ ]:
set_seed()
N_BOOTSTRAP = 10
gene_shap_records = {gene : [] for gene in genes_expression}

event_idx = np.where(train_events_np == 1)[0]
no_event_idx = np.where(train_events_np == 0)[0]
for i in range(N_BOOTSTRAP):
    idx_ev = resample(event_idx, random_state=i, replace=True)
    idx_no = resample(no_event_idx, random_state=i, replace=True)
    idx = np.concatenate([idx_ev, idx_no])
    
    X_boot = train_X_np[idx]
    dur_boot = train_durations_np[idx]
    ev_boot = train_events_np[idx]
    
    ev_boot_idx = np.where(ev_boot == 1)[0]
    no_ev_boot_idx = np.where(ev_boot == 0)[0]
    
    val_ev = resample(ev_boot_idx,    n_samples=max(3, int(0.15*len(ev_boot_idx))),    random_state=i, replace=False)
    val_no = resample(no_ev_boot_idx, n_samples=max(5, int(0.15*len(no_ev_boot_idx))), random_state=i, replace=False)
    val_idx = np.concatenate([val_ev, val_no])
    train_idx = np.setdiff1d(np.arange(len(X_boot)), val_idx)
    
    
    X_tr, X_val = X_boot[train_idx], X_boot[val_idx]
    dur_tr, dur_val = dur_boot[train_idx], dur_boot[val_idx]
    ev_tr,  ev_val = ev_boot[train_idx], ev_boot[val_idx]
    
    net_MLPVanilla = tt.torchtuples.practical.MLPVanilla(
        in_features=in_features,
        num_nodes=[16, 16],
        dropout=0.2,
        batch_norm=True,
        output_bias=False,
        out_features=1
    )
    
    #DeepSurv
    model_boot = CoxPH(
        net_MLPVanilla,
        tt.torchtuples.optim.Adam(
            lr=0.0005,
            weight_decay=0.001 # type: ignore
        )
    )
    model_boot.fit(
        X_tr,
        (dur_tr, ev_tr),
        batch_size=256,
        epochs=200,
        verbose=False,
        callbacks=[tt.torchtuples.callbacks.EarlyStopping(patience=10)],
        val_data=(X_val, (dur_val, ev_val))
    )
    
    def predict_fn(x):
        x_tensor = torch.tensor(x, dtype=torch.float32)
        with torch.no_grad():
            out = model_boot.net(x_tensor)
        return out.cpu().numpy().reshape(-1)


    preds_check = predict_fn(X_boot[:5])
    if np.isnan(preds_check).any():
        print(f"  → Bootstrap {i+1} divergió, saltando")
        continue

    background = shap.sample(X_boot, 50)
    explainer = shap.explainers.Permutation(
        predict_fn, background,
        feature_names=genes_expression,
        max_evals=2001
    )
    shap_values = explainer(test_X_np)
    mean_abs = np.abs(shap_values.values).mean(axis=0)

    for j, gene in enumerate(genes_expression):
        gene_shap_records[gene].append(mean_abs[j])

    print(f"Bootstrap {i+1}/{N_BOOTSTRAP} OK")

PermutationExplainer explainer: 105it [02:55,  1.79s/it]                         


Bootstrap 1/10 OK


PermutationExplainer explainer: 105it [02:55,  1.79s/it]                         


Bootstrap 2/10 OK


PermutationExplainer explainer: 105it [02:55,  1.79s/it]                         


...

Bootstrap 8/10 OK


PermutationExplainer explainer: 105it [02:57,  1.81s/it]                         


Bootstrap 9/10 OK


PermutationExplainer explainer: 105it [02:59,  1.83s/it]                         

Bootstrap 10/10 OK


In [70]:
gene_shap_records["ABI2"]


[]

In [ ]:
valid_genes = [g for g in genes_expression if len(gene_shap_records[g]) > 0]

records   = np.array([gene_shap_records[g] for g in valid_genes])   # (genes, n_boot)
mean_shap = records.mean(axis=1)
std_shap  = records.std(axis=1)

# Estabilidad real: coeficiente de variación (menor = más estable)
cv_shap = np.where(mean_shap > 0, std_shap / mean_shap, np.nan)

# Frecuencia de aparición en el top-50 a lo largo de los bootstraps
TOP_K = 50
topk_freq = np.zeros(len(valid_genes))
for b in range(records.shape[1]):
    top_idx = np.argsort(-records[:, b])[:TOP_K]
    topk_freq[top_idx] += 1
topk_freq /= max(records.shape[1], 1)

bootstrap_df = pd.DataFrame({
    'gene':      valid_genes,
    'mean_shap': mean_shap,
    'std_shap':  std_shap,
    'cv_shap':   cv_shap,
    'topk_freq': topk_freq,
}).sort_values('mean_shap', ascending=False)

boot_to_csv = bootstrap_df.head(100)
boot_to_csv.to_csv("boot_strap_df_deepsurv.csv")

In [72]:
bootstrap_df

,gene,mean_shap,std_shap,stability


### CoxNett DeppSurv

In [ ]:
set_seed()
N_BOOTSTRAP = 10
gene_shap_records_coxnnet = {gene : [] for gene in genes_expression}

event_idx = np.where(train_events_np == 1)[0]
no_event_idx = np.where(train_events_np == 0)[0]
for i in range(N_BOOTSTRAP):
    idx_ev = resample(event_idx, random_state=i, replace=True)
    idx_no = resample(no_event_idx, random_state=i, replace=True)
    idx = np.concatenate([idx_ev, idx_no])
    
    X_boot = train_X_np[idx]
    dur_boot = train_durations_np[idx]
    ev_boot = train_events_np[idx]
    
    ev_boot_idx = np.where(ev_boot == 1)[0]
    no_ev_boot_idx = np.where(ev_boot == 0)[0]
    
    val_ev = resample(ev_boot_idx,    n_samples=max(3, int(0.15*len(ev_boot_idx))),    random_state=i, replace=False)
    val_no = resample(no_ev_boot_idx, n_samples=max(5, int(0.15*len(no_ev_boot_idx))), random_state=i, replace=False)
    val_idx = np.concatenate([val_ev, val_no])
    train_idx = np.setdiff1d(np.arange(len(X_boot)), val_idx)
    
    
    X_tr, X_val = X_boot[train_idx], X_boot[val_idx]
    dur_tr, dur_val = dur_boot[train_idx], dur_boot[val_idx]
    ev_tr,  ev_val = ev_boot[train_idx], ev_boot[val_idx]
    
    net_MLPVanilla = tt.torchtuples.practical.MLPVanilla(
        in_features=in_features,
        num_nodes=[32],
        dropout=0.2,
        batch_norm=True,
        output_bias=False,
        out_features=1
    )
    
    #DeepSurv
    model_boot = CoxPH(
        net_MLPVanilla,
        tt.torchtuples.optim.Adam(
            lr=0.0001,
            weight_decay=0.0001 # type: ignore
        )
    )
    model_boot.fit(
        X_tr,
        (dur_tr, ev_tr),
        batch_size=256,
        epochs=200,
        verbose=False,
        callbacks=[tt.torchtuples.callbacks.EarlyStopping(patience=10)],
        val_data=(X_val, (dur_val, ev_val))
    )
    
    def predict_fn(x):
        x_tensor = torch.tensor(x, dtype=torch.float32)
        with torch.no_grad():
            out = model_boot.net(x_tensor)
        return out.cpu().numpy().reshape(-1)


    preds_check = predict_fn(X_boot[:5])
    if np.isnan(preds_check).any():
        print(f"  → Bootstrap {i+1} divergió, saltando")
        continue

    background = shap.sample(X_boot, 50)
    explainer = shap.explainers.Permutation(
        predict_fn, background,
        feature_names=genes_expression,
        max_evals=2001
    )
    shap_values = explainer(test_X_np)
    mean_abs = np.abs(shap_values.values).mean(axis=0)

    for j, gene in enumerate(genes_expression):
        gene_shap_records_coxnnet[gene].append(mean_abs[j])

    print(f"Bootstrap {i+1}/{N_BOOTSTRAP} OK")

PermutationExplainer explainer: 105it [03:02,  1.84s/it]                         


Bootstrap 1/10 OK


PermutationExplainer explainer: 105it [03:05,  1.88s/it]                         


Bootstrap 2/10 OK


PermutationExplainer explainer: 105it [02:59,  1.84s/it]                         


...

Bootstrap 8/10 OK


PermutationExplainer explainer: 105it [03:01,  1.83s/it]                         


Bootstrap 9/10 OK


PermutationExplainer explainer: 105it [02:59,  1.83s/it]                         

Bootstrap 10/10 OK


In [ ]:
valid_genes = [g for g in gene_shap_records_coxnnet if len(gene_shap_records_coxnnet[g]) > 0]

records   = np.array([gene_shap_records_coxnnet[g] for g in valid_genes])   # (genes, n_boot)
mean_shap = records.mean(axis=1)
std_shap  = records.std(axis=1)

# Estabilidad real: coeficiente de variación (menor = más estable)
cv_shap = np.where(mean_shap > 0, std_shap / mean_shap, np.nan)

# Frecuencia de aparición en el top-50 a lo largo de los bootstraps
TOP_K = 50
topk_freq = np.zeros(len(valid_genes))
for b in range(records.shape[1]):
    top_idx = np.argsort(-records[:, b])[:TOP_K]
    topk_freq[top_idx] += 1
topk_freq /= max(records.shape[1], 1)

bootstrap_df_coxnnet = pd.DataFrame({
    'gene':      valid_genes,
    'mean_shap': mean_shap,
    'std_shap':  std_shap,
    'cv_shap':   cv_shap,
    'topk_freq': topk_freq,
}).sort_values('mean_shap', ascending=False)

boot_to_csv_coxnnet = bootstrap_df_coxnnet.head(100)
boot_to_csv_coxnnet.to_csv("boot_strap_df_coxnnet.csv")

In [69]:
boot_to_csv_coxnnet

,gene,mean_shap,std_shap,stability
923,SLC25A28,0.040470,0.009723,1.0
943,PDZD11,0.040112,0.006498,1.0
955,TNK2,0.039840,0.008473,1.0
659,GUCY1A3,0.039814,0.008265,1.0
640,QARS1,0.039664,0.005210,1.0
...,...,...,...,...
178,MRFAP1L1,0.037377,0.005627,1.0
805,RBM4B,0.037362,0.008208,1.0
529,PPA1,0.037352,0.005516,1.0
450,SGPP1,0.037342,0.005541,1.0
